# C4-classical-ml-practice — Practice p11 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
FEATURES = ["length_mm", "width_mm", "mass_g", "moisture_pct"]
ks = np.array([1, 3, 5, 7, 9, 11])
beans = pd.read_csv("data/beans.csv")
X = beans[FEATURES].to_numpy(dtype=float)
y = beans["species"].to_numpy()
X_tmp, X_te, y_tmp, y_te = train_test_split(
    X, y, test_size=18, random_state=SEED, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tmp, y_tmp, test_size=18, random_state=SEED, stratify=y_tmp
)
val_results = []
for k in ks:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ])
    candidate.fit(X_tr, y_tr)
    val_results.append(candidate.score(X_val, y_val))
val_accs = np.array(val_results, dtype=float)
best_k = int(ks[np.argmax(val_accs)])
X_refit = np.vstack([X_tr, X_val])
y_refit = np.concatenate([y_tr, y_val])
best_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
])
best_pipe.fit(X_refit, y_refit)
test_acc = float(best_pipe.score(X_te, y_te))

val_accs, best_k, test_acc

Candidate pipelines are fit only on the 54 training rows and ranked solely by their 18-row validation accuracies. Because the k array is ascending, the first maximum selects k = 1 over the tied k = 5; only then is a fresh pipeline fit on all 72 non-test rows and evaluated once, yielding 17/18 test accuracy.

### Answer check

In [ ]:
expected_val_accs = np.array([17/18, 15/18, 17/18, 16/18, 16/18, 16/18])
assert val_accs.shape == (6,) and np.allclose(val_accs, expected_val_accs)
assert best_k == 1 and isinstance(best_k, int)
assert X_tr.shape[0] == 54 and X_val.shape[0] == 18 and X_te.shape[0] == 18
assert np.isclose(test_acc, 17/18)